# Does fusing CLIP+reactivity-delta, DINOv2, and AEROBLADE actually help?

A local CPU-only version of this experiment on 17 test images was inconclusive --
too small to distinguish real signal from noise (a single flipped pair changes
AUROC by ~1.4 points at that scale). This notebook reruns it properly, on GPU,
against the **full 443-image held-out test split** (the same deterministic split
in `noise-residual-experiment/model/test_manifest.json`, disjoint from the
organizer's WildFake demonstration set), with DINOv2 trained on the full 2506
train images (matching CLIP's training scale -- the earlier small run showed a
thin-data DINOv2 looks weaker than it really is).

Three independently-scored experts, fused via proper stratified K-fold cross-
validation (not a single train/test split, and not the leave-one-out toy version
from the CPU run):
1. **CLIP + reactivity-delta** -- the existing trained production model, reused as-is
   (no retraining -- it's already trained on the full 2506 train images)
2. **DINOv2** linear probe, trained here on the full train split
3. **AEROBLADE** reconstruction distance -- training-free, scored directly

**Before running:** Settings -> Accelerator -> GPU T4 x2 (this notebook pins
`machine_shape: NvidiaTeslaT4` in its metadata already -- a P100 crashes with
`no kernel image is available for execution on the device`, which is what killed
the first attempt at this). Attach three datasets: `byteprint-realdata`,
`byteprint-code`, `clip-reactivity-code`.

In [ ]:
import time
t0 = time.time()
def checkpoint(label):
    print(f'[+{time.time()-t0:7.1f}s] {label}', flush=True)

checkpoint('notebook start')

import torch
assert torch.cuda.is_available(), 'no GPU visible -- check Settings > Accelerator'
cap = torch.cuda.get_device_capability()
print(f'GPU: {torch.cuda.get_device_name()}  compute capability {cap}')
assert cap >= (7, 0), (
    f'compute capability {cap} is too old for this PyTorch build (needs >=7.0) -- '
    'this is the P100-vs-T4 problem from before. Check machine_shape in kernel-metadata.json.'
)
checkpoint('GPU check passed')

In [ ]:
import glob, os, shutil

# Both code repos are private on GitHub -- an anonymous `git clone` in a batch
# Kaggle kernel just hangs forever on a username prompt (no stdin/TTY). Pull
# from Kaggle Datasets instead, same fix as the earlier diagnostic notebook.
def find_code(marker_file, dest):
    candidates = [
        os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/{marker_file}', recursive=True)
    ]
    assert candidates, f'attach the dataset containing {marker_file} to this notebook'
    shutil.copytree(candidates[0], dest)
    return dest

byteprint_dir = find_code('pyproject.toml', '/kaggle/working/byteprint')
clip_dir = find_code('production_pipeline.py', '/kaggle/working/clip_src')
checkpoint(f'copied byteprint source -> {byteprint_dir}, clip source -> {clip_dir}')

%cd /kaggle/working/byteprint
!pip install -q -e .
assert _exit_code == 0, f'byteprint install failed, exit code {_exit_code}'
!pip install -q diffusers lpips accelerate safetensors transformers scikit-image
assert _exit_code == 0, f'pip install failed, exit code {_exit_code}'
# specialists.pkl/domain_classifier.pkl were pickled with scikit-learn 1.9.0 (LogisticRegression
# dropped the `multi_class` attribute in a way that breaks loading on Kaggle's older default
# sklearn -- AttributeError: 'LogisticRegression' object has no attribute 'multi_class'). Pin the
# exact version that pickled them rather than fight sklearn's pickle forward-compatibility.
!pip install -q scikit-learn==1.9.0
assert _exit_code == 0, f'scikit-learn pin failed, exit code {_exit_code}'
checkpoint('pip installs done')

In [ ]:
import sys
sys.path.insert(0, clip_dir)
sys.path.insert(0, byteprint_dir)

# Fail fast on any missing dependency, before spending GPU time on extraction.
try:
    import production_pipeline  # noqa: F401
    import byteprint.cli  # noqa: F401
except ImportError as e:
    raise RuntimeError(f'import sanity check failed: {e}') from e
checkpoint('import sanity check passed (production_pipeline + byteprint.cli both import cleanly)')

In [ ]:
train_candidates = [
    p for p in glob.glob('/kaggle/input/**/train', recursive=True)
    if os.path.isdir(os.path.join(p, 'real')) and os.path.isdir(os.path.join(p, 'fake'))
]
test_candidates = [
    p for p in glob.glob('/kaggle/input/**/test', recursive=True)
    if os.path.isdir(os.path.join(p, 'real')) and os.path.isdir(os.path.join(p, 'fake'))
]
assert train_candidates and test_candidates, 'attach the byteprint-realdata dataset first'
train_dir, test_dir = train_candidates[0], test_candidates[0]

!find {train_dir} -type f | wc -l
!find {test_dir} -type f | wc -l
checkpoint(f'found data: train={train_dir} test={test_dir}')

## 1. CLIP + reactivity-delta -- score the full 443-image test set

Reuses the already-trained production model (domain classifier + 5 domain
specialists) as-is. No training here, just inference.

In [ ]:
import sys, json
sys.path.insert(0, clip_dir)
os.chdir(clip_dir)  # production_pipeline.py's MODEL_DIR is relative to its own file, this is just belt-and-braces

from PIL import Image
from production_pipeline import predict_proba

def collect_split(root):
    records = []
    real_dir = os.path.join(root, 'real')
    for f in sorted(os.listdir(real_dir)):
        records.append((os.path.join(real_dir, f), 0))
    fake_root = os.path.join(root, 'fake')
    for source in sorted(os.listdir(fake_root)):
        for f in sorted(os.listdir(os.path.join(fake_root, source))):
            records.append((os.path.join(fake_root, source, f), 1))
    return records

test_records = collect_split(test_dir)
print(f'{len(test_records)} test images')

paths = [r[0] for r in test_records]
labels = [r[1] for r in test_records]

clip_scores = []
chunk = 64
for i in range(0, len(paths), chunk):
    batch_paths = paths[i:i+chunk]
    imgs = [Image.open(p).convert('RGB') for p in batch_paths]
    probs, _ = predict_proba(imgs, batch_size=32, model_dir=os.path.join(clip_dir, 'model'))
    clip_scores.extend(float(p) for p in probs)
    if (i // chunk) % 2 == 0:
        checkpoint(f'CLIP scored {i+len(batch_paths)}/{len(paths)}')

assert len(clip_scores) == len(paths)
clip_out = [{'path': p, 'label': l, 'clip_score': s} for p, l, s in zip(paths, labels, clip_scores)]
with open('/kaggle/working/clip_scores.json', 'w') as f:
    json.dump(clip_out, f, indent=2)
checkpoint(f'CLIP scoring done, {len(clip_out)} images -> clip_scores.json')
os.chdir('/kaggle/working/byteprint')

## 2. DINOv2 -- train on the full 2506 train images, score the full 443 test images

This is the step that only used 150 train images in the CPU experiment. On GPU
the full 2506 should still be fast.

In [ ]:
!python -m byteprint.cli extract --data {train_dir} --cache /kaggle/working/cache/dino_train \
    --expert dinov2 --augment 2 --device cuda --batch-size 64 --seed 0
assert _exit_code == 0, f'dinov2 train extract failed, exit code {_exit_code}'
checkpoint('dinov2 train extract done (full 2506 images)')

!python -m byteprint.cli extract --data {test_dir} --cache /kaggle/working/cache/dino_test \
    --expert dinov2 --device cuda --batch-size 64
assert _exit_code == 0, f'dinov2 test extract failed, exit code {_exit_code}'
checkpoint('dinov2 test extract done (full 443 images, clean only)')

In [ ]:
!python -m byteprint.cli train --cache /kaggle/working/cache/dino_train \
    --out /kaggle/working/runs/probe.joblib --target-fpr 0.01
assert _exit_code == 0, f'probe training failed, exit code {_exit_code}'
checkpoint('dinov2 probe trained')

In [ ]:
sys.path.insert(0, '/kaggle/working/byteprint')
from byteprint.cache import EmbeddingStore, ExtractConfig
from byteprint.probe import LinearProbe

config = ExtractConfig(**json.loads(open('/kaggle/working/cache/dino_test/config.json').read()))
store = EmbeddingStore.open('/kaggle/working/cache/dino_test', config)
probe = LinearProbe.load('/kaggle/working/runs/probe.joblib')

features = store.matrix()
scores = probe.score(features)
dino_labels = store.labels()
dino_paths = store.paths()

dino_out = [{'path': p, 'label': int(l), 'dino_score': float(s)}
            for p, l, s in zip(dino_paths, dino_labels, scores)]
with open('/kaggle/working/dino_scores.json', 'w') as f:
    json.dump(dino_out, f, indent=2)
checkpoint(f'DINOv2 scoring done, {len(dino_out)} images -> dino_scores.json')

## 3. AEROBLADE -- training-free, score the full 443 test images

Measured throughput on the earlier successful diagnostic run: ~17.5 crops/sec
on a T4. 443 images x 4 crops = ~1772 crops -> projected ~1.7 minutes.

In [ ]:
_recon_t0 = time.time()
!python -m byteprint.cli extract --data {test_dir} --cache /kaggle/working/cache/recon_test \
    --expert recon --device cuda --batch-size 8
assert _exit_code == 0, f'recon test extract failed, exit code {_exit_code}'
checkpoint(f'AEROBLADE extract done in {time.time()-_recon_t0:.1f}s (full 443 images, clean only)')

In [ ]:
from byteprint.recon import aeroblade_score

config = ExtractConfig(**json.loads(open('/kaggle/working/cache/recon_test/config.json').read()))
store = EmbeddingStore.open('/kaggle/working/cache/recon_test', config)

distances = store.matrix()
scores = aeroblade_score(distances)
aero_labels = store.labels()
aero_paths = store.paths()

aero_out = [{'path': p, 'label': int(l), 'aeroblade_score': float(s)}
            for p, l, s in zip(aero_paths, aero_labels, scores)]
with open('/kaggle/working/aeroblade_scores.json', 'w') as f:
    json.dump(aero_out, f, indent=2)
checkpoint(f'AEROBLADE scoring done, {len(aero_out)} images -> aeroblade_scores.json')

## 4. Fusion -- does combining all three actually beat the best single expert?

Proper 5-fold stratified cross-validation this time (N=443, not the N=17 LOOCV
toy version) -- each fold's fusion weights are fit only on the other folds, so
the reported fused AUROC is a genuine held-out estimate, not circular.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

clip_map = {r['path']: r['clip_score'] for r in json.load(open('/kaggle/working/clip_scores.json'))}
dino_map = {r['path']: r['dino_score'] for r in json.load(open('/kaggle/working/dino_scores.json'))}
aero_map = {r['path']: r['aeroblade_score'] for r in json.load(open('/kaggle/working/aeroblade_scores.json'))}
label_map = {r['path']: r['label'] for r in json.load(open('/kaggle/working/clip_scores.json'))}

shared_paths = sorted(set(clip_map) & set(dino_map) & set(aero_map))
print(f'shared images across all 3 scores: {len(shared_paths)} (expect close to 443 -- '
      f'a big gap means the path strings did not match between experts, check dataset dirs)')

y = np.array([label_map[p] for p in shared_paths])
X_clip = np.array([clip_map[p] for p in shared_paths])
X_dino = np.array([dino_map[p] for p in shared_paths])
X_aero = np.array([aero_map[p] for p in shared_paths])
X_all = np.column_stack([X_clip, X_dino, X_aero])

print(f'label balance: {int((y==0).sum())} real, {int((y==1).sum())} fake')

def cv_auc(X, y, n_splits=5, seed=0):
    """Stratified K-fold: for a single column this just measures the raw score's
    AUROC on each fold's held-out slice (no fitting needed, reported for symmetry).
    For multiple columns, fits+evaluates a fresh logistic regression per fold."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_aucs = []
    oof_preds = np.zeros(len(y))
    for train_idx, test_idx in skf.split(X, y):
        if X.ndim == 1 or X.shape[1] == 1:
            col = X if X.ndim == 1 else X[:, 0]
            oof_preds[test_idx] = col[test_idx]
            fold_aucs.append(roc_auc_score(y[test_idx], col[test_idx]))
        else:
            scaler = StandardScaler().fit(X[train_idx])
            clf = LogisticRegression(max_iter=2000).fit(scaler.transform(X[train_idx]), y[train_idx])
            p = clf.predict_proba(scaler.transform(X[test_idx]))[:, 1]
            oof_preds[test_idx] = p
            fold_aucs.append(roc_auc_score(y[test_idx], p))
    overall_auc = roc_auc_score(y, oof_preds)
    return overall_auc, np.array(fold_aucs), oof_preds

print()
print(f'{"score":<22}{"pooled AUROC":>14}{"fold mean":>12}{"fold std":>10}')
results = {}
for name, X in [
    ('CLIP+reactivity', X_clip),
    ('DINOv2', X_dino),
    ('AEROBLADE', X_aero),
    ('fused: all 3', X_all),
    ('fused: clip+dino', X_all[:, [0,1]]),
    ('fused: clip+aero', X_all[:, [0,2]]),
    ('fused: dino+aero', X_all[:, [1,2]]),
]:
    overall, folds, oof = cv_auc(X, y)
    results[name] = (overall, folds, oof)
    print(f'{name:<22}{overall:>14.4f}{folds.mean():>12.4f}{folds.std():>10.4f}')

checkpoint('fusion cross-validation done')

In [ ]:
# Error analysis: where does the best single expert disagree most with the full fusion?
best_single_name = max(['CLIP+reactivity', 'DINOv2', 'AEROBLADE'], key=lambda n: results[n][0])
best_single_oof = results[best_single_name][2]
fused_oof = results['fused: all 3'][2]

disagreement = np.abs(fused_oof - best_single_oof)
top_disagree_idx = np.argsort(-disagreement)[:10]

print(f'best single expert: {best_single_name} (AUROC {results[best_single_name][0]:.4f})')
print(f'fused (all 3): AUROC {results["fused: all 3"][0]:.4f}')
print()
print('top 10 cases where fusion disagreed most with the best single expert:')
print(f'{"label":<6}{"clip":>8}{"dino":>8}{"aero":>8}{"fused":>8}{"single":>8}  path')
for i in top_disagree_idx:
    p = shared_paths[i]
    print(f'{y[i]:<6}{X_clip[i]:>8.3f}{X_dino[i]:>8.3f}{X_aero[i]:>8.3f}'
          f'{fused_oof[i]:>8.3f}{best_single_oof[i]:>8.3f}  {os.path.basename(p)}')

checkpoint('error analysis done')

In [ ]:
summary = {
    name: {'pooled_auc': float(overall), 'fold_mean': float(folds.mean()), 'fold_std': float(folds.std())}
    for name, (overall, folds, oof) in results.items()
}
with open('/kaggle/working/fusion_results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

!tar -czf /kaggle/working/fusion_experiment_results.tar.gz -C /kaggle/working \
    clip_scores.json dino_scores.json aeroblade_scores.json fusion_results_summary.json runs
print('done -- download fusion_experiment_results.tar.gz from the notebook output panel')
checkpoint('results packaged')